# Semana 11 — IBM Quantum en Hardware Real
## Ejecutar circuitos en un computador cuántico de verdad

**Objetivo:** Conectar con IBM Quantum, ejecutar circuitos en hardware real, comparar con el simulador, y entender las limitaciones del hardware NISQ (Noisy Intermediate-Scale Quantum).

**Hito verificable:** Ejecutas un estado de Bell en hardware real de IBM y comparas los resultados con el simulador ideal, analizando el ruido observado.

> **Prerrequisito:** Tener una cuenta en https://quantum.ibm.com/ y haber ejecutado `python configurar_ibm.py` (ver Semana 1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

# IBM Quantum Runtime
try:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.compiler import transpile
    IBM_DISPONIBLE = True
    print('qiskit-ibm-runtime importado correctamente')
except ImportError:
    IBM_DISPONIBLE = False
    print('qiskit-ibm-runtime no instalado — solo modo simulador')
    print('Instala con: pip install qiskit-ibm-runtime')

print('Qiskit Aer disponible para simulaciones locales')

## Ejercicio 11.1 — Conectar con IBM Quantum

In [ ]:
if IBM_DISPONIBLE:
    # Cargar credenciales guardadas (requiere haber ejecutado configurar_ibm.py)
    try:
        service = QiskitRuntimeService()
        backends = service.backends(simulator=False, operational=True)
        print(f'Backends disponibles ({len(backends)}):')
        for b in sorted(backends, key=lambda x: x.num_qubits):
            print(f'  {b.name:30s}  {b.num_qubits:3d} qubits  '
                  f'pendientes: {b.status().pending_jobs}')
    except Exception as e:
        print(f'Error de conexión: {e}')
        print('Ejecuta primero: python configurar_ibm.py')
else:
    print('Ejecutando en modo simulador local (AerSimulator)')

## Ejercicio 11.2 — Elegir el backend y transpilar

In [ ]:
# Circuito de Bell para ejecutar en hardware
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

print('Circuito original:')
print(qc_bell.draw(output='text'))

if IBM_DISPONIBLE:
    try:
        # Seleccionar el backend con menos cola
        backend = min(
            [b for b in backends if b.num_qubits >= 2],
            key=lambda b: b.status().pending_jobs
        )
        print(f'\nBackend seleccionado: {backend.name}')
        
        # Transpilar al conjunto de puertas nativas del hardware
        qc_transpilado = transpile(qc_bell, backend=backend, optimization_level=3)
        print(f'\nCircuito transpilado:')
        print(qc_transpilado.draw(output='text'))
        print(f'Profundidad original: {qc_bell.depth()}')
        print(f'Profundidad transpilada: {qc_transpilado.depth()}')
        print(f'Puertas: {qc_transpilado.count_ops()}')
    except Exception as e:
        print(f'Error al transpilar: {e}')
else:
    print('\nSin IBM: transpilación simulada para base cx, rz, sx, x')
    from qiskit.compiler import transpile
    qc_transpilado = transpile(qc_bell, basis_gates=['cx', 'rz', 'sx', 'x'],
                               optimization_level=3)
    print(qc_transpilado.draw(output='text'))

## Ejercicio 11.3 — Ejecutar en hardware y comparar con simulador

In [ ]:
# Simulador ideal (referencia)
sim_ideal = AerSimulator()
counts_ideal = sim_ideal.run(qc_bell, shots=4000).result().get_counts()
print('=== Simulador ideal ===')
print(f'Resultados: {counts_ideal}')
print(f'Estados de error (01, 10): 0 (teórico)')

if IBM_DISPONIBLE:
    try:
        # Ejecutar en hardware real
        print('\n=== Hardware IBM Quantum ===')
        print(f'Enviando job a {backend.name}...')
        sampler = Sampler(backend)
        job = sampler.run([qc_transpilado], shots=4000)
        print(f'Job ID: {job.job_id()}')
        print('Esperando resultado... (puede tardar varios minutos en la cola)')
        result = job.result()
        counts_hardware = result[0].data.c.get_counts()
        
        errores_hw = counts_hardware.get('01', 0) + counts_hardware.get('10', 0)
        print(f'Resultados hardware: {counts_hardware}')
        print(f'Estados de error (01, 10): {errores_hw} ({100*errores_hw/4000:.1f}%)')
        
    except Exception as e:
        print(f'Error al ejecutar en hardware: {e}')
        print('\nCargando resultados de ejemplo (de una ejecución real típica):')
        counts_hardware = {'00': 1850, '11': 1820, '01': 175, '10': 155}
        errores_hw = counts_hardware.get('01', 0) + counts_hardware.get('10', 0)
        print(f'Resultados (ejemplo): {counts_hardware}')
        print(f'Estados de error: {errores_hw} ({100*errores_hw/4000:.1f}%)')
else:
    print('\n=== Simulador con ruido (emulando hardware) ===')
    from qiskit_aer.noise import NoiseModel, depolarizing_error
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.003, 1), ['h', 'rz', 'sx'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
    sim_ruidoso = AerSimulator(noise_model=noise_model)
    counts_hw_sim = sim_ruidoso.run(qc_transpilado, shots=4000).result().get_counts()
    errores = counts_hw_sim.get('01', 0) + counts_hw_sim.get('10', 0)
    print(f'Resultados: {counts_hw_sim}')
    print(f'Tasa de error simulada: {100*errores/4000:.1f}%')

## Ejercicio 11.4 — Propiedades del hardware: mapa de conectividad

In [ ]:
if IBM_DISPONIBLE:
    try:
        # Propiedades del backend
        props = backend.properties()
        coupling = backend.coupling_map
        
        print(f'Backend: {backend.name}')
        print(f'Qubits: {backend.num_qubits}')
        print(f'Puertas nativas: {backend.operation_names}')
        print()
        
        # T1, T2 y tasa de error de los primeros qubits
        print(f'{"Qubit":>7}  {"T1(μs)":>10}  {"T2(μs)":>10}  {"Error readout":>14}')
        print('-' * 50)
        for i in range(min(5, backend.num_qubits)):
            try:
                t1 = props.t1(i) * 1e6
                t2 = props.t2(i) * 1e6
                readout_err = props.readout_error(i)
                print(f'{i:>7}  {t1:>10.1f}  {t2:>10.1f}  {readout_err:>14.4f}')
            except Exception:
                print(f'{i:>7}  (datos no disponibles)')
    except Exception as e:
        print(f'Error al obtener propiedades: {e}')
else:
    print('Datos típicos de IBM Quantum (ibm_sherbrooke, 127 qubits):')
    print()
    print(f'{"Qubit":>7}  {"T1(μs)":>10}  {"T2(μs)":>10}  {"Error readout":>14}')
    print('-' * 50)
    datos_tipicos = [(0, 189, 142, 0.012), (1, 203, 158, 0.008),
                     (2, 175, 133, 0.015), (3, 221, 167, 0.009), (4, 198, 145, 0.011)]
    for q, t1, t2, err in datos_tipicos:
        print(f'{q:>7}  {t1:>10.1f}  {t2:>10.1f}  {err:>14.4f}')
    print()
    print('Conectividad: no todos los qubits están conectados entre sí.')
    print('Un CNOT entre qubits no adyacentes requiere puertas SWAP adicionales.')

## Ejercicio 11.5 — Mitigación de errores básica

In [ ]:
# Mitigación de errores de readout (medición)
# Idea: caracterizar la matriz de confusión y corregir los resultados

def matriz_confusion_simulada(p_readout_error=0.03):
    """Simula la matriz de confusión del readout para 1 qubit."""
    # M[i,j] = P(medir i | estado j)
    M = np.array([
        [1 - p_readout_error, p_readout_error],
        [p_readout_error, 1 - p_readout_error]
    ])
    return M

def mitigar_conteos(counts_ruidosos, M, n_shots):
    """Aplica la mitigación de errores de readout."""
    # Vector de frecuencias medidas
    freq = np.array([counts_ruidosos.get('0', 0), counts_ruidosos.get('1', 0)]) / n_shots
    # Invertir la matriz de confusión
    M_inv = np.linalg.inv(M)
    freq_mitigada = M_inv @ freq
    freq_mitigada = np.clip(freq_mitigada, 0, 1)  # Proyectar a probabilidades válidas
    freq_mitigada /= freq_mitigada.sum()  # Renormalizar
    return {'0': int(freq_mitigada[0] * n_shots), '1': int(freq_mitigada[1] * n_shots)}

# Simular medición de |0⟩ con error de readout
p_err = 0.05
M = matriz_confusion_simulada(p_err)
n_shots = 10000

# Estado real: |0⟩ → debería medir siempre '0'
counts_ruidosos = {'0': int(n_shots * (1 - p_err)), '1': int(n_shots * p_err)}
counts_mitigados = mitigar_conteos(counts_ruidosos, M, n_shots)

print(f'Error de readout: {p_err}')
print(f'Sin mitigación: {counts_ruidosos}')
print(f'Con mitigación: {counts_mitigados}')
print()
print('La mitigación de errores es una herramienta esencial en el era NISQ.')
print('Qiskit Runtime incluye M3 (Matrix-free Measurement Mitigation) automáticamente.')

## Mini reto — Completar antes de avanzar a la Semana 12

Si tienes acceso a IBM Quantum:
1. Ejecuta el circuito GHZ de 3 qubits: |GHZ⟩ = (|000⟩+|111⟩)/√2
2. Compara resultados en el simulador ideal, simulador con ruido, y hardware real
3. Calcula la fidelidad de estado usando tomografía de estado (máximo likelihood)

Si no tienes acceso al hardware:
1. Implementa la tomografía de estado completa para 2 qubits midiendo en las bases X, Y, Z
2. Usa los resultados para reconstruir la matriz de densidad
3. Verifica con el estado de Bell ideal

In [ ]:
# TU CÓDIGO AQUÍ
def crear_ghz(n_qubits: int) -> QuantumCircuit:
    """Crea el estado GHZ de n qubits."""
    # TODO: implementar
    pass

def tomografia_estado_2q(counts_XY, counts_XZ, counts_YZ, counts_ZZ, n_shots=8192):
    """Reconstruye la matriz de densidad de 2 qubits."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 11

Antes de avanzar, comprueba que puedes:

- [ ] Conectar con IBM Quantum (o entender por qué y cómo hacerlo)
- [ ] Transpilar un circuito al conjunto de puertas de un backend real
- [ ] Ejecutar en hardware y comparar con el simulador
- [ ] Interpretar las propiedades del backend (T1, T2, error rates)
- [ ] Aplicar mitigación básica de errores de readout

## Reflexión (escribe aquí tu respuesta)

**¿Qué diferencia hay entre ejecutar en IBM Quantum con el Sampler vs. el Estimator?**

*(tu respuesta aquí)*

**¿Qué es la era NISQ y qué la diferencia de la computación cuántica tolerante a fallos?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 12 — Proyecto Final*